In [ ]:
import dynamiqs as dq
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.optimize import least_squares
import sys
import os
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python/")
import Plot_setting
from Plot_setting import *
# --- constants (computed once) ---
ħ = 1
e = 1
phi_quantum = 1/2

#w_c = 15 * 2 * np.pi
#phi_c = 0.3
w_a = 3.823578966821696*2*np.pi#3.85
w_b = 6.697567050165361*2*np.pi#6.761
w_c = 15.0*2*np.pi
# Ej = 27.148
# Delta_Ej = 0.316
# phi_zpf_mem = 0.139
# phi_zpf_buf = 0.204
# phi_zpf_ats = 0.1
E_J = 26*2*np.pi
phi_a = 0.071
phi_b = 0.171
phi_c = 0.3


n_a = 6
n_b = 6
n_c = 5
a = dq.destroy(n_a)
b = dq.destroy(n_b)
c = dq.destroy(n_c)
varphi_a_op_tot = dq.tensor(phi_a*(a + a.dag()), dq.eye(n_b), dq.eye(n_c))
varphi_b_op_tot = dq.tensor(dq.eye(n_a), phi_b*(b + b.dag()), dq.eye(n_c))
varphi_c_op_tot = dq.tensor(dq.eye(n_a), dq.eye(n_b), phi_c*(c + c.dag()))

H_base = dq.tensor(w_a * dq.number(n_a), dq.eye(n_b), dq.eye(n_c)) \
    + dq.tensor(dq.eye(n_a), w_b * dq.number(n_b), dq.eye(n_c)) \
    + dq.tensor(dq.eye(n_a), dq.eye(n_b),  w_c * dq.number(n_c))

In [ ]:
phi_delta_list = np.linspace(2.5, 3.0, 30)
transitions_list = []
matrix_elements_list = []
drive_op_np = (1j*dq.tensor(dq.eye(n_a), dq.dag(b)-b, dq.eye(n_c))).to_numpy()#(1j*dq.tensor(dq.eye(n_a), dq.eye(n_b), dq.dag(c)-c)).to_numpy()#
n_state = 3
for phi_delta in phi_delta_list:
    print(phi_delta/2/np.pi)
    phi_op = varphi_a_op_tot + varphi_b_op_tot + varphi_c_op_tot + phi_delta * dq.eye_like(varphi_a_op_tot)
    H = H_base - 2 * E_J * dq.cosm(phi_op)
    H = H.to_numpy()

    evals, evecs = sp.sparse.linalg.eigsh(H, k=n_state, which='SM', tol = 0, maxiter=1e6)
    evals = np.real(evals)
    evals = np.sort(evals)
    transitions = evals - evals[0]   # spectroscopy from ground state
    transitions_list.append(transitions[:n_state])
    ground = evecs[:, 0]
    matrix_elements_list.append(np.array([
        np.abs(evecs[:,k].conj() @ drive_op_np @ ground) ** 2
        for k in range(n_state)])
    )

transitions_arr = np.array(transitions_list)
matrix_elements_arr = np.array(matrix_elements_list)
best_idx = np.argmax(matrix_elements_arr, axis=1) 
best_idx[:20] = 2*np.ones(20)         # shape (n,)
best_idx[-20:] = 2*np.ones(20)         # shape (n,)
transitions_filtered = transitions_arr[np.arange(len(transitions_arr)), best_idx]  # shape (n,)

In [ ]:
# Normalize across all phi and all bands
drive_op_min = matrix_elements_arr.min()
drive_op_max = matrix_elements_arr.max()
alphas_norm = (matrix_elements_arr - drive_op_min) / (drive_op_max - drive_op_min + 1e-12)

fig, ax = plt.subplots(figsize=(8, 5))

for k in range(2, n_state):   # skip ground state (always 0)
    for i in range(len(phi_delta_list) - 1):
        ax.plot(
            phi_delta_list[i:i+2],
            transitions_arr[i:i+2, k]/ (2*np.pi),
            color='steelblue',
            alpha=float(alphas_norm[i, k]),
            linewidth=1.5,
            solid_capstyle='round',
        )
plt.scatter(phi_delta_list, transitions_filtered/2/np.pi)
ax.set_xlabel(r'$\phi_\delta$')
ax.set_ylabel('Energy ' + r"$\omega/2\pi$"+' (GHz)')
ax.set_title('Spectroscopy \n transparency ∝ drive operator matrix element (ground to excited)')

# Colorbar as a proxy for alpha scale
sm = plt.cm.ScalarMappable(
    cmap=mcolors.LinearSegmentedColormap.from_list('alpha_cmap', ['white', 'steelblue']),
    norm=plt.Normalize(vmin=drive_op_min, vmax=drive_op_max)
)
sm.set_array([])
fig.colorbar(sm, ax=ax, label=r'$| \langle k| i(b^\dagger - b)|0 \rangle| ^2$')

plt.tight_layout()
plt.show()

In [ ]:
phi_delta_list

In [ ]:
freqs_file = "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python/FIF/Fit _data_16_06_2026/data/freqs_Hz_zoom2.npy"
dc_delta_file = "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python/FIF/Fit _data_16_06_2026/data/dc_delta_zoom2.npy"
data_file = "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python/FIF/Fit _data_16_06_2026/data/data_zoom2.npy"

# %%
data = np.load(data_file)
low_freq_cutoff = 2000
high_freq_cutoff = 800
data = data.T[::-1, :][high_freq_cutoff:low_freq_cutoff, :84]
freqs_Hz = np.load(freqs_file)[high_freq_cutoff:low_freq_cutoff]
dc_delta = np.load(dc_delta_file)[:84]

idx_points_fit = []
max_transmission_freq_array=[]
freq_Hz_inverted = freqs_Hz[::-1]
for phi_delta in phi_delta_list:
    idx = np.argmin(np.abs(dc_delta-phi_delta))
    #print(dc_delta[idx])
    max_transmission_freq_array.append(freq_Hz_inverted[np.argmax(data[:,idx])]/1e9)
max_transmission_freq_array = np.array(max_transmission_freq_array)

array_fit = np.copy(max_transmission_freq_array)
array_fit[1] = 7.41
array_fit[3] = 7.38
#array_fit[26] = 6.56
#array_fit[30:33] = [7.1,7.05, 7.12]
#array_fit[36] = 6.6
#array_fit[-1] = 7.4
#%% Plotting experimental data and crosses for the selected points
plt.figure(figsize=(6, 4))
plt.imshow(
    data,
    aspect='auto',
    extent=(dc_delta[0], dc_delta[-1], freqs_Hz[0] / 1e9, freqs_Hz[-1] / 1e9),      
)
plt.scatter(phi_delta_list, array_fit)
#plt.axvline(np.pi+0.01)
plt.colorbar(label='Transmission (a.u.)')
plt.xlabel(r'$\varphi_{\Delta}$ (rad)')
plt.ylabel('Frequency (GHz)')
plt.title('SCT-0006-03-A sys 2 - Experimental Data')
plt.ylim(freqs_Hz[0] / 1e9, freqs_Hz[-1] / 1e9)
plt.tight_layout()
plt.show()


In [ ]:
def compute_transitions(params, phi_delta_list):
    w_c, phi_c = params
    H_base = dq.tensor(w_a * dq.number(n_a), dq.eye(n_b), dq.eye(n_c)) \
    + dq.tensor(dq.eye(n_a), w_b * dq.number(n_b), dq.eye(n_c)) \
    + dq.tensor(dq.eye(n_a), dq.eye(n_b),  w_c * dq.number(n_c))
    varphi_a_op_tot = dq.tensor(phi_a*(a + a.dag()), dq.eye(n_b), dq.eye(n_c))
    varphi_b_op_tot = dq.tensor(dq.eye(n_a), phi_b*(b + b.dag()), dq.eye(n_c))
    varphi_c_op_tot = dq.tensor(dq.eye(n_a), dq.eye(n_b), phi_c*(c + c.dag()))

    transitions_list = []

    for phi_delta in phi_delta_list:
        phi_delta_shift = phi_delta-0.01
        phi_op = varphi_a_op_tot + varphi_b_op_tot + varphi_c_op_tot + phi_delta_shift * dq.eye_like(varphi_a_op_tot)

        H = H_base - 2 * E_J * dq.cosm(phi_op)
        H = H.to_numpy()

        evals, _ = sp.sparse.linalg.eigsh(H, k=n_state, which='SM', tol = 0, maxiter=1e6)
        evals = np.real(evals)
        evals = np.sort(evals)
        transitions = evals - evals[0]   # spectroscopy from ground state
        transitions_list.append(transitions[:n_state])
    transitions_arr = np.array(transitions_list)
    transitions_filtered = transitions_arr[np.arange(len(transitions_arr)), best_idx]  # shape (n,)
    return transitions_filtered

def residuals(params, phi_delta_list, data):
    model = compute_transitions(params, phi_delta_list)
    return (model - data).ravel()

params0 = [15*2*np.pi, 0.3]   # initial guess
result = least_squares(residuals, params0, args=(phi_delta_list, array_fit), max_nfev=5e3)
params_fit = result.x

print("Fitted parameters:", params_fit)
print("Cost:", result.cost)

In [ ]:
phi_a = 7.09733964e-02
phi_b = 1.71006336e-01
phi_c = 2.99979580e-01


n_a = 12
n_b = 6
n_c = 5
a = dq.destroy(n_a)
b = dq.destroy(n_b)
c = dq.destroy(n_c)
varphi_a_op_tot = dq.tensor(phi_a*(a + a.dag()), dq.eye(n_b), dq.eye(n_c))
varphi_b_op_tot = dq.tensor(dq.eye(n_a), phi_b*(b + b.dag()), dq.eye(n_c))
varphi_c_op_tot = dq.tensor(dq.eye(n_a), dq.eye(n_b), phi_c*(c + c.dag()))

H_base = dq.tensor(w_a * dq.number(n_a), dq.eye(n_b), dq.eye(n_c)) \
    + dq.tensor(dq.eye(n_a), w_b * dq.number(n_b), dq.eye(n_c)) \
    + dq.tensor(dq.eye(n_a), dq.eye(n_b),  w_c * dq.number(n_c))
phi_delta_list = np.linspace(2.5, 3.75, 60)
transitions_list = []
matrix_elements_list = []
drive_op_np = (1j*dq.tensor(dq.eye(n_a), dq.dag(b)-b, dq.eye(n_c))).to_numpy()#(1j*dq.tensor(dq.eye(n_a), dq.eye(n_b), dq.dag(c)-c)).to_numpy()#
n_state = 4
for phi_delta in phi_delta_list:
    print(phi_delta/2/np.pi)
    phi_delta_shift = phi_delta-0.01
    phi_op = varphi_a_op_tot + varphi_b_op_tot + varphi_c_op_tot + phi_delta_shift * dq.eye_like(varphi_a_op_tot)
    H = H_base - 2 * E_J * dq.cosm(phi_op)
    H = H.to_numpy()

    evals, evecs = sp.sparse.linalg.eigsh(H, k=n_state, which='SM', tol = 0, maxiter=1e6)
    evals = np.real(evals)
    evals = np.sort(evals)
    transitions = evals - evals[0]   # spectroscopy from ground state
    transitions_list.append(transitions[:n_state])
    ground = evecs[:, 0]
    matrix_elements_list.append(np.array([
        np.abs(evecs[:,k].conj() @ drive_op_np @ ground) ** 2
        for k in range(n_state)])
    )

transitions_arr = np.array(transitions_list)
matrix_elements_arr = np.array(matrix_elements_list)

In [ ]:
# Normalize across all phi and all bands
drive_op_min = matrix_elements_arr.min()
drive_op_max = matrix_elements_arr.max()
alphas_norm = (matrix_elements_arr - drive_op_min) / (drive_op_max - drive_op_min + 1e-12)

fig, ax = plt.subplots(figsize=(8, 5))

for k in range(2, n_state):   # skip ground state (always 0)
    for i in range(len(phi_delta_list) - 1):
        ax.plot(
            phi_delta_list[i:i+2],
            transitions_arr[i:i+2, k]/ (2*np.pi),
            color='steelblue',
            alpha=float(alphas_norm[i, k]),
            linewidth=1.5,
            solid_capstyle='round',
        )
ax.set_xlabel(r'$\phi_\delta$')
ax.set_ylabel('Energy ' + r"$\omega/2\pi$"+' (GHz)')
ax.set_title('Spectroscopy \n transparency ∝ drive operator matrix element (ground to excited)')

# Colorbar as a proxy for alpha scale
sm = plt.cm.ScalarMappable(
    cmap=mcolors.LinearSegmentedColormap.from_list('alpha_cmap', ['white', 'steelblue']),
    norm=plt.Normalize(vmin=drive_op_min, vmax=drive_op_max)
)
sm.set_array([])
fig.colorbar(sm, ax=ax, label=r'$| \langle k| i(b^\dagger - b)|0 \rangle| ^2$')

plt.tight_layout()
plt.show()